# BE-01 · How a backend really works

[Open in Colab](https://colab.research.google.com/github/BuiltBySwap/engineering-lab/blob/main/docs/topics/BE-01.ipynb)

**Status card** (edit it; the Progress and Tags pages are built from it):

```yaml
id: BE-01
title: "How a backend really works"
track: backend
date: 2026-10-01
status: done  # not-started | in-progress | partial | done
score: "4/5"
insight: "A server is a long-running process that owns a port, accepts TCP connections, reads HTTP text and writes bytes back; Ktor (framework) and Netty (engine) do the plumbing so I only write routes."
tags: [BE-01, backend, ktor, http, sockets]
pending: []
```

## Goal
Understand what happens on the server side of a Retrofit call, from socket to response.

## What I learned (plain words)
- A **process** owns a **port**. The IP address finds the machine, the port finds the process.
- A server is a loop: `accept()` a connection, read the request line and headers, write a response, repeat.
- **Stateless:** each request stands alone, so identity travels with every request (the `Authorization` header).
- **Engine vs framework:** Netty handles sockets and threads, Ktor gives routing and plugins, my code is the handlers.
- One thread per connection does not scale. A few threads plus one coroutine per request does.

## Examples
Raw socket server, one thread per client:

```kotlin
fun main() {
    val server = ServerSocket(8080)
    while (true) {
        val socket = server.accept()
        thread { handle(socket) }
    }
}
```

What `curl -v` showed (request line, headers, blank line, then the response):

```text
> GET /hello HTTP/1.1
< HTTP/1.1 200 OK
< Content-Length: 24
Hello from a raw socket!
```

## Bugs and fixes
| Error or symptom | Cause | Fix |
|---|---|---|
| `Connection refused` | Nothing is listening on that port | Start the server |
| `Address already in use` | Another process owns the port | `lsof -nP -iTCP:8080 -sTCP:LISTEN`, then `kill <PID>` |
| `Cannot access java.lang.Object` in IntelliJ | Gradle was not using a JDK | Set Gradle JVM to JDK 21 |
| Second client waited | One thread did everything | Handle each connection on its own thread |

## Quiz and mistakes
- Score: 4/5.
- A port is not part of the IP address.
- Swapping Netty for CIO also changes the dependency, not only the import and `mainClass`.
- Coroutines are not threads: they share a few threads and suspend instead of blocking.

## Origin story
1. Problem: as a mobile developer I only saw the client side of HTTP.
2. Old way: write raw socket servers by hand.
3. Why it broke: one thread per client cannot scale, and there is no routing, JSON, TLS or keep-alive.
4. The invention: engines (Netty, CIO) and frameworks (Ktor), with a coroutine per request.
5. How it works inside: OS port, `accept()`, parse request, route, handler, respond.
6. Where I've used it: every Retrofit/OkHttp call at work.
7. Trade-offs: raw sockets give control but do not scale; more threads are not more scale.

## In my own words
✏️ Two lines.

## Next
[BE-02](BE-02.md): Ktor fundamentals.